# Module 05 — System Benchmark (The Final Flight Check)
**Objective:** Execute a holistic performance review of the entire Pricing Engine.


In [2]:
# %% [markdown]
# # Module 05 — System Benchmark (The Final Flight Check)
# **Objective:** Execute a holistic performance review of the entire Pricing Engine.

# %% [code]
import sys, os, logging
import pandas as pd
import tensorflow as tf
import numpy as np

# Add project root to path
sys.path.append(os.path.abspath(".."))

from pricing_engine.data_loader import load_and_clean_seattle_data
from pricing_engine.demand_model import DemandModel
from pricing_engine.pricing_strategy import *
from pricing_engine.benchmark import PricingBenchmarkSuite

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("Benchmark")

# --- 1. Load System (Robust Mode) ---
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

# --- DEFINING THE LOADER (Previously missing) ---
class FastKerasWrapper:
    def __init__(self, keras_model, features):
        self.model = keras_model
        self.features = features
    def predict(self, df):
        input_dict = {
            name: tf.convert_to_tensor(df[name].values.reshape(-1, 1))
            for name in self.features if name in df.columns
        }
        return self.model(input_dict, training=False).numpy().flatten()

def load_brain_optimized():
    ARTIFACT_DIR = "../demand_artifacts"
    MODELS = {}
    model_map = {
        ModelRole.UNCERTAINTY: "HierarchicalBayes.pkl",
        ModelRole.MEAN: "LGBM_Tweedie.pkl",
        ModelRole.SAFETY: "TF_Lattice.pkl"
    }
    
    logger.info("🔌 Booting System Models...")
    for role, filename in model_map.items():
        path = os.path.join(ARTIFACT_DIR, filename)
        try:
            if "Lattice" in role:
                import tensorflow_lattice as tfl
                art_dir = path + "_artifacts" 
                if not os.path.exists(art_dir): art_dir = path.replace(".pkl", "") + "_artifacts"
                keras_path = os.path.join(art_dir, "keras_model")
                
                # Robust TFLattice Loading
                try:
                    if hasattr(tfl, 'custom_objects'):
                        k_model = tf.keras.models.load_model(keras_path, custom_objects=tfl.custom_objects())
                    elif hasattr(tfl, 'custom_objects_scope'):
                        with tfl.custom_objects_scope():
                            k_model = tf.keras.models.load_model(keras_path)
                    else:
                        k_model = tf.keras.models.load_model(keras_path)
                except:
                    k_model = tf.keras.models.load_model(keras_path)
                    
                MODELS[role] = FastKerasWrapper(k_model, [l.name for l in k_model.inputs])
                logger.info(f"   ✅ {role} Online (Fast Inference)")
            else:
                MODELS[role] = DemandModel.load(path)
                logger.info(f"   ✅ {role} Online (Standard)")
        except Exception as e:
            logger.error(f"   ❌ {role} Failed: {e}")
            if role == ModelRole.MEAN: raise
            
    return MODELS

# Initialize the models
MODELS = load_brain_optimized() 

# --- 2. Data Prep (Smart Feature Engineering) ---
logger.info("🌍 Loading Live Context...")

# 1. Load & Merge (The base data)
df_raw = load_and_clean_seattle_data("../data/calendar.csv", "../data/listings.csv")

# 2. Time-Based Aggregation (Weekly)
# We aggregate by week because that's the resolution of our demand models
audit_df = (
    df_raw
    .assign(week_date=pd.to_datetime(df_raw["date"]).dt.to_period("W").dt.start_time)
    .groupby(["listing_id", "week_date"])
    .agg({
        "is_booked_proxy": "max", 
        "price": "mean",
        "accommodates": "first", 
        "bedrooms": "first", 
        "bathrooms": "first",       # <--- Metadata
        "neighborhood": "first",    # <--- Metadata
        "room_type": "first"        # <--- Metadata
    }).reset_index()
    .rename(columns={"is_booked_proxy": "is_booked", "price": "avg_price"})
)

# 3. Feature Generation (The "Why are we doing this?" part)
# We MUST generate these because the trained LGBM/Tweedie model expects them.
audit_df["log_price"] = np.log1p(audit_df["avg_price"])
audit_df["week_of_year"] = audit_df["week_date"].dt.isocalendar().week.astype(int)
audit_df["month"] = audit_df["week_date"].dt.month
audit_df["neighborhood"] = audit_df["neighborhood"].fillna("Unknown").astype("category")
audit_df["room_type"] = audit_df["room_type"].fillna("Entire home/apt").astype("category")

# 4. Fill Missing Metadata 
# If a listing is missing 'bathrooms', we fill with median to stop crashes
for c in ["accommodates", "bedrooms", "bathrooms"]: 
    if c in audit_df.columns:
        audit_df[c] = audit_df[c].fillna(audit_df[c].median())

# 5. Sanitization
audit_df = audit_df.dropna(subset=["avg_price", "is_booked"])

# 6. Train/Test Split
split_idx = int(len(audit_df) * 0.8)
ref_df = audit_df.iloc[:split_idx]
curr_df = audit_df.iloc[split_idx:]

logger.info(f"   ✅ Reference Set: {len(ref_df):,} rows")
logger.info(f"   ✅ Current Set:   {len(curr_df):,} rows")

# --- 3. Run Benchmark Suite ---
logger.info("🚀 Starting System Benchmark Suite...")
policy = ThompsonSamplingPolicy()

suite = PricingBenchmarkSuite(policy, MODELS)
results_df = suite.run_all(ref_df, curr_df)

# --- 4. Scorecard ---
def color_status(val):
    color = 'green' if val == 'PASS' else 'red'
    if val == 'WARN': color = 'orange'
    return f'color: {color}; font-weight: bold'

print("\n🏆 SYSTEM HEALTH CERTIFICATE 🏆")
display(results_df.style.map(color_status, subset=['status']))

# --- 5. Final Decision ---
if (results_df['status'] == 'FAIL').sum() == 0:
    print("\n🟢 SYSTEM STATUS: OPERATIONAL. READY FOR DEPLOYMENT.")
else:
    print("\n🔴 SYSTEM STATUS: DEGRADED. DEPLOYMENT BLOCKED.")

2026-01-10 04:38:06,765 | INFO | 🔌 Booting System Models...


2026-01-10 04:38:06,778 | INFO |    ✅ HierarchicalBayes Online (Standard)
2026-01-10 04:38:06,810 | INFO |    ✅ LGBM_Tweedie Online (Standard)
2026-01-10 04:38:07,534 | INFO |    ✅ TF_Lattice Online (Fast Inference)
2026-01-10 04:38:07,535 | INFO | 🌍 Loading Live Context...
2026-01-10 04:38:07,536 | INFO | Loading raw data...
2026-01-10 04:38:12,950 | INFO | Total rows loaded: 1393570
2026-01-10 04:38:12,961 | INFO | Action observed rate: 67.06%
2026-01-10 04:38:12,964 | INFO | Exposure rate: 67.06%
2026-01-10 04:38:12,966 | INFO | Booked proxy rate: 32.94%
2026-01-10 04:38:14,982 | INFO |    ✅ Reference Set: 112,864 rows
2026-01-10 04:38:14,987 | INFO |    ✅ Current Set:   28,216 rows
2026-01-10 04:38:14,990 | INFO | 🚀 Starting System Benchmark Suite...


TypeError: SafetyGovernor.validate_and_clamp() got an unexpected keyword argument 'price'